In [1]:

import os
import sys
import csv
import time
import math
import copy
import random
import shutil
import tempfile
import argparse
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torch.optim.swa_utils import AveragedModel, SWALR
from torchvision import transforms

BASE_DIR = "shift-guard-10-robust-image-classification-challenge"
OUT_DIR  = "."

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]
NAME_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
IDX_TO_NAME = {i: n for n, i in NAME_TO_IDX.items()}
NUM_CLASSES  = 10

MU  = (0.4914, 0.4822, 0.4465)
SIG = (0.2470, 0.2435, 0.2616)

# ---- Fix randomness (important for reproducibility) ----
def seed_everything(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class Meter:
    def __init__(self):
        self.val = self.avg = self.total = self.n = 0

    def update(self, v, c=1):
        self.val    = v
        self.total += v * c
        self.n     += c
        self.avg    = self.total / self.n


def compute_f1(preds, targets):
    from sklearn.metrics import f1_score
    return f1_score(targets, preds, average="macro", zero_division=0)


def class_report(preds, targets):
    from sklearn.metrics import classification_report
    return classification_report(targets, preds, target_names=CLASS_NAMES, zero_division=0)


def ckpt_ok(path):
    import zipfile
    if not os.path.isfile(path):
        return False
    try:
        with zipfile.ZipFile(path) as z:
            return len(z.namelist()) > 0
    except Exception:
        return False


def atomic_save(obj, path):
    d = os.path.dirname(path) or "."
    os.makedirs(d, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=d, suffix=".tmp")
    try:
        os.close(fd)
        torch.save(obj, tmp)
        shutil.move(tmp, path)
    except Exception:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


class PatchMask:
    def __init__(self, size=16):
        self.size = size

    def __call__(self, t):
        h, w = t.size(1), t.size(2)
        m = torch.ones(h, w, dtype=t.dtype)
        cy = random.randint(0, h - 1)
        cx = random.randint(0, w - 1)
        r0 = max(0, cy - self.size // 2); r1 = min(h, cy + self.size // 2)
        c0 = max(0, cx - self.size // 2); c1 = min(w, cx + self.size // 2)
        m[r0:r1, c0:c1] = 0.0
        return t * m.unsqueeze(0)


def train_tf():
    return transforms.Compose([
        transforms.RandomCrop(32, padding=4, fill=128),
        transforms.RandomHorizontalFlip(),
        transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
        transforms.ToTensor(),
        transforms.Normalize(MU, SIG),
        PatchMask(size=16),
    ])


def eval_tf():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MU, SIG),
    ])


def aug_tf():
    return transforms.Compose([
        transforms.RandomCrop(32, padding=4, fill=128),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        transforms.RandomRotation(10, fill=128),
        transforms.ToTensor(),
        transforms.Normalize(MU, SIG),
    ])


# Dataset Class 

class CifarDataset(Dataset):
    def __init__(self, root, mode="train", transform=None, holdout=0.05, seed=42):
        self.root      = root
        self.mode      = mode
        self.transform = transform

        if mode in ("train", "val"):
            csv_path = os.path.join(root, "train_labels.csv")
            ids, lbls = [], []
            with open(csv_path) as f:
                for row in csv.DictReader(f):
                    ids.append(row["id"].strip().zfill(6))
                    lbls.append(row["label"].strip())

            rng = np.random.RandomState(seed)
            buckets = {c: [] for c in CLASS_NAMES}
            for i, l in enumerate(lbls):
                buckets[l].append(i)

            tr_idx, va_idx = [], []
            for c in CLASS_NAMES:
                b = buckets[c][:]
                rng.shuffle(b)
                n = max(1, int(len(b) * holdout))
                va_idx.extend(b[:n])
                tr_idx.extend(b[n:])

            sel = tr_idx if mode == "train" else va_idx
            self.img_ids = [ids[i]               for i in sel]
            self.targets = [NAME_TO_IDX[lbls[i]] for i in sel]
            self.img_dir = os.path.join(root, "train_images")

        elif mode == "test":
            sub = os.path.join(root, "sample_submission.csv")
            self.img_ids = []
            self.targets = None
            with open(sub) as f:
                for row in csv.DictReader(f):
                    self.img_ids.append(row["id"].strip().zfill(6))
            self.img_dir = os.path.join(root, "test_images")

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        fid = self.img_ids[idx]
        img = Image.open(os.path.join(self.img_dir, f"{fid}.png")).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.targets is not None:
            return img, self.targets[idx]
        return img, fid

    def class_counts(self):
        return np.bincount(self.targets, minlength=NUM_CLASSES)

    def balanced_sampler(self):
        cnt = np.bincount(self.targets, minlength=NUM_CLASSES)
        w   = 1.0 / (np.sqrt(cnt) + 1e-6)
        sw  = [w[t] for t in self.targets]
        return WeightedRandomSampler(sw, len(self.targets), replacement=True)


class WideBlock(nn.Module):
    def __init__(self, in_c, out_c, stride, drop=0.3):
        super().__init__()
        self.bn1  = nn.BatchNorm2d(in_c)
        self.cv1  = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn2  = nn.BatchNorm2d(out_c)
        self.cv2  = nn.Conv2d(out_c, out_c, 3, stride=1, padding=1, bias=False)
        self.drop = nn.Dropout(p=drop) if drop > 0 else nn.Identity()
        self.sc   = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.sc = nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False)

    def forward(self, x):
        h = self.cv1(F.relu(self.bn1(x), inplace=True))
        h = self.drop(h)
        h = self.cv2(F.relu(self.bn2(h), inplace=True))
        return h + self.sc(x)


# WideResNet 

class WideNet(nn.Module):
    def __init__(self, depth=28, widen=10, nc=10, drop=0.3):
        super().__init__()
        assert (depth - 4) % 6 == 0
        nb  = (depth - 4) // 6
        chs = [16, 16 * widen, 32 * widen, 64 * widen]

        self.stem   = nn.Conv2d(3, chs[0], 3, stride=1, padding=1, bias=False)
        self.grp1   = self._make_group(chs[0], chs[1], nb, 1, drop)
        self.grp2   = self._make_group(chs[1], chs[2], nb, 2, drop)
        self.grp3   = self._make_group(chs[2], chs[3], nb, 2, drop)
        self.bn_out = nn.BatchNorm2d(chs[3])
        self.fc     = nn.Linear(chs[3], nc)
        self._init()

    def _make_group(self, ic, oc, n, s, d):
        layers = [WideBlock(ic, oc, s, d)]
        for _ in range(1, n):
            layers.append(WideBlock(oc, oc, 1, d))
        return nn.Sequential(*layers)

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.grp1(x)
        x = self.grp2(x)
        x = self.grp3(x)
        x = F.relu(self.bn_out(x), inplace=True)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)


class AdjustedCE(nn.Module):
    def __init__(self, counts, smooth=0.0):
        super().__init__()
        f = torch.tensor(counts, dtype=torch.float32)
        f = f / f.sum()
        self.register_buffer("lp", torch.log(f + 1e-12))
        self.smooth = smooth

    def forward(self, logits, targets):
        return F.cross_entropy(logits + self.lp.unsqueeze(0), targets, label_smoothing=self.smooth)


def do_mixup(x, y, a=1.0):
    lam  = np.random.beta(a, a) if a > 0 else 1.0
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam


def do_cutmix(x, y, a=1.0):
    lam  = np.random.beta(a, a) if a > 0 else 1.0
    perm = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape
    rw = int(W * math.sqrt(1.0 - lam)); rh = int(H * math.sqrt(1.0 - lam))
    cx = random.randint(0, W - 1);      cy = random.randint(0, H - 1)
    x0 = max(0, cx - rw // 2); x1 = min(W, cx + rw // 2)
    y0 = max(0, cy - rh // 2); y1 = min(H, cy + rh // 2)
    out = x.clone()
    out[:, :, y0:y1, x0:x1] = x[perm, :, y0:y1, x0:x1]
    lam = 1 - (y1 - y0) * (x1 - x0) / (W * H)
    return out, y, y[perm], lam


def mix_loss(crit, logits, ya, yb, lam):
    return lam * crit(logits, ya) + (1 - lam) * crit(logits, yb)


# Training for one epoch

def train_one_epoch(model, loader, crit, opt, scaler, device, mp=0.5):
    model.train()
    meter = Meter()
    ok = tot = 0

    for imgs, labs in loader:
        imgs, labs = imgs.to(device), labs.to(device)
        mixed = False
        opt.zero_grad()

        with torch.amp.autocast('cuda'):
            if mp > 0 and random.random() < mp:
                if random.random() < 0.5:
                    imgs, ya, yb, lam = do_mixup(imgs, labs)
                else:
                    imgs, ya, yb, lam = do_cutmix(imgs, labs)
                out   = model(imgs)
                loss  = mix_loss(crit, out, ya, yb, lam)
                mixed = True
            else:
                out  = model(imgs)
                loss = crit(out, labs)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt); scaler.update()

        meter.update(loss.item(), imgs.size(0))
        if not mixed:
            ok  += out.argmax(1).eq(labs).sum().item()
            tot += labs.size(0)

    return meter.avg, 100.0 * ok / tot if tot > 0 else 0.0


@torch.no_grad()
def eval_one_epoch(model, loader, crit, device):
    model.eval()
    meter = Meter()
    ps, ls = [], []

    for imgs, labs in loader:
        imgs, labs = imgs.to(device), labs.to(device)
        out  = model(imgs)
        loss = crit(out, labs)
        meter.update(loss.item(), imgs.size(0))
        ps.extend(out.argmax(1).cpu().numpy())
        ls.extend(labs.cpu().numpy())

    f1  = compute_f1(ps, ls)
    acc = 100.0 * np.mean(np.array(ps) == np.array(ls))
    return meter.avg, acc, f1, ps, ls

# I have added the checkpoint rule like if in case training stop then checkpoints will be saved.
def train_model(seed, cfg, device, freq):
    seed_everything(seed)
    print(f"\n{'─'*60}\n  Seed {seed} | Epochs {cfg.epochs}\n{'─'*60}\n")

    tr_ds = CifarDataset(BASE_DIR, "train", train_tf(), holdout=cfg.val_ratio, seed=42)
    va_ds = CifarDataset(BASE_DIR, "val",   eval_tf(),  holdout=cfg.val_ratio, seed=42)

    if cfg.debug:
        tr_ds = Subset(tr_ds, range(min(200, len(tr_ds))))
        va_ds = Subset(va_ds, range(min(50,  len(va_ds))))

    sampler = tr_ds.balanced_sampler() if (not cfg.debug and hasattr(tr_ds, "balanced_sampler")) else None
    nw      = 2 if not cfg.debug else 0
    pf      = 2 if nw > 0 else None

    tr_ld = DataLoader(tr_ds, batch_size=cfg.batch_size,
                       shuffle=(sampler is None), sampler=sampler,
                       num_workers=nw, pin_memory=True, drop_last=True,
                       persistent_workers=False, prefetch_factor=pf)
    va_ld = DataLoader(va_ds, batch_size=cfg.batch_size * 2,
                       shuffle=False, num_workers=nw, pin_memory=True,
                       persistent_workers=False, prefetch_factor=pf)

    model = WideNet(depth=28, widen=10, nc=NUM_CLASSES, drop=0.3).to(device)
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)

    sm   = 0.0 if cfg.mix_prob > 0 else cfg.label_smoothing
    crit = AdjustedCE(freq, smooth=sm).to(device)
    opt  = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd, eps=1e-4)

    wu = 5
    def sched_fn(ep):
        if ep < wu:
            return (ep + 1) / wu
        t = (ep - wu) / (cfg.epochs - wu)
        return 0.5 * (1 + math.cos(math.pi * t))
    lrsched = torch.optim.lr_scheduler.LambdaLR(opt, sched_fn)

    do_swa = cfg.swa_start < cfg.epochs
    if do_swa:
        swa_model = AveragedModel(model)
        swa_sched = SWALR(opt, swa_lr=cfg.swa_lr)

    scaler   = torch.amp.GradScaler('cuda')
    best_f1  = 0.0
    best_wts = None
    no_imp   = 0

    ckpt_path = f"checkpoints/run_{seed}_latest.pth"
    swa_path  = f"checkpoints/run_{seed}_swa.pth"
    os.makedirs("checkpoints", exist_ok=True)
    start_ep  = 0

    if ckpt_ok(ckpt_path):
        print(f"  Resuming: {ckpt_path}")
        ck  = torch.load(ckpt_path, map_location=device, weights_only=False)
        raw = model.module if isinstance(model, torch.nn.DataParallel) else model
        raw.load_state_dict(ck["model"])
        opt.load_state_dict(ck["optimizer"])
        lrsched.load_state_dict(ck["scheduler"])
        start_ep = ck["epoch"] + 1
        best_f1  = ck["best_f1"]
        no_imp   = ck.get("no_imp", 0)
        if ck.get("best_wts") is not None:
            best_wts = ck["best_wts"]
        print(f"  epoch={start_ep}/{cfg.epochs}  best_f1={best_f1:.4f}")
        if do_swa and start_ep >= cfg.swa_start and ckpt_ok(swa_path):
            print(f"  Restoring SWA: {swa_path}")
            swa_model.load_state_dict(torch.load(swa_path, map_location=device, weights_only=False))
    elif os.path.isfile(ckpt_path):
        print("  Corrupt checkpoint — starting fresh.")
        os.remove(ckpt_path)
    else:
        print("  No checkpoint found — starting fresh.")

    for ep in range(start_ep, cfg.epochs):
        t0 = time.time()
        tl, ta = train_one_epoch(model, tr_ld, crit, opt, scaler, device, cfg.mix_prob)

        in_swa = do_swa and ep >= cfg.swa_start
        if in_swa:
            swa_model.update_parameters(model)
            swa_sched.step()
        else:
            lrsched.step()

        vl, va, vf, _, _ = eval_one_epoch(model, va_ld, crit, device)
        lr_now = opt.param_groups[0]["lr"]
        tag    = " [SWA]" if in_swa else ""
        print(f"  Ep{ep+1:3d}/{cfg.epochs} | "
              f"Tr {tl:.3f}/{ta:.1f}% | "
              f"Va {vl:.3f}/{va:.1f}% F1={vf:.4f} | "
              f"lr={lr_now:.6f} | {time.time()-t0:.1f}s{tag}")

        if vf > best_f1:
            best_f1  = vf
            no_imp   = 0
            raw      = model.module if isinstance(model, torch.nn.DataParallel) else model
            best_wts = copy.deepcopy(raw.state_dict())
        else:
            no_imp += 1

        if (not in_swa) and no_imp >= cfg.early_stop_patience:
            print(f"  Early stop at epoch {ep+1} ({no_imp} epochs no gain)")
            raw = model.module if isinstance(model, torch.nn.DataParallel) else model
            atomic_save({"epoch": ep, "model": raw.state_dict(), "optimizer": opt.state_dict(),
                         "scheduler": lrsched.state_dict(), "best_f1": best_f1,
                         "best_wts": best_wts, "no_imp": no_imp}, ckpt_path)
            break

        raw = model.module if isinstance(model, torch.nn.DataParallel) else model
        atomic_save({"epoch": ep, "model": raw.state_dict(), "optimizer": opt.state_dict(),
                     "scheduler": lrsched.state_dict(), "best_f1": best_f1,
                     "best_wts": best_wts, "no_imp": no_imp}, ckpt_path)

        if in_swa:
            atomic_save(swa_model.state_dict(), swa_path)

    if do_swa:
        print("\n  BN update for SWA...")
        bn_ld = DataLoader(tr_ds, batch_size=cfg.batch_size * 2,
                           shuffle=True, num_workers=0, pin_memory=False)
        torch.optim.swa_utils.update_bn(bn_ld, swa_model, device=device)
        del bn_ld

        _, _, sf, _, _ = eval_one_epoch(swa_model, va_ld, crit, device)
        print(f"  SWA F1={sf:.4f}  prev_best={best_f1:.4f}")
        if sf > best_f1:
            best_f1  = sf
            inner    = getattr(swa_model.module if hasattr(swa_model, "module") else swa_model, "module", swa_model)
            best_wts = copy.deepcopy(inner.state_dict())

    out_path = f"checkpoints/final_{seed}.pth"
    atomic_save(best_wts, out_path)
    print(f"  Saved → {out_path}")

    for p in [ckpt_path, swa_path]:
        if os.path.isfile(p):
            os.remove(p)

    return best_wts, best_f1


@torch.no_grad()
def run_tta(model, ds, device, views=12, bs=512):
    model.eval()
    nw = 2

    ds.transform = eval_tf()
    base_ld = DataLoader(ds, batch_size=bs, shuffle=False,
                         num_workers=nw, pin_memory=True, persistent_workers=False)
    acc, ids = [], []
    for imgs, sid in base_ld:
        acc.append(F.softmax(model(imgs.to(device)), dim=1).cpu())
        ids.extend(sid)
    probs = torch.cat(acc, dim=0)

    for v in range(views):
        ds.transform = aug_tf()
        ld = DataLoader(ds, batch_size=bs, shuffle=False,
                        num_workers=nw, pin_memory=True, persistent_workers=False)
        vp = []
        for imgs, _ in ld:
            vp.append(F.softmax(model(imgs.to(device)), dim=1).cpu())
        probs += torch.cat(vp, dim=0)
        print(f"    view {v+1}/{views}", end="\r")

    print()
    return probs / (views + 1), ids


# Generate submission file

def make_submission(probs, ids, path):
    preds  = probs.argmax(dim=1).numpy()
    labels = [IDX_TO_NAME[p] for p in preds]
    with open(path, "w", newline="") as f:
        f.write("id,label\n")
        for sid, lbl in zip(ids, labels):
            f.write(f"{sid},{lbl}\n")
    tally = Counter(labels)
    print(f"\n  Written → {path}")
    for c in CLASS_NAMES:
        print(f"    {c:12s}: {tally.get(c, 0):5d}")
    return labels

# main ebtry point
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--debug",               action="store_true")
    ap.add_argument("--gpu",                 type=int,   default=0)
    ap.add_argument("--data-root",           type=str,   default=None)
    ap.add_argument("--epochs",              type=int,   default=109)
    ap.add_argument("--swa-start",           type=int,   default=95)
    ap.add_argument("--seeds",               type=int,   nargs="+", default=[42])
    ap.add_argument("--batch-size",          type=int,   default=64)
    ap.add_argument("--lr",                  type=float, default=1.5e-3)
    ap.add_argument("--wd",                  type=float, default=1e-4)
    ap.add_argument("--val-ratio",           type=float, default=0.05)
    ap.add_argument("--swa-lr",              type=float, default=5e-4)
    ap.add_argument("--tta",                 type=int,   default=12)
    ap.add_argument("--mix-prob",            type=float, default=0.5)
    ap.add_argument("--label-smoothing",     type=float, default=0.1)
    ap.add_argument("--output",              type=str,   default="submission.csv")
    ap.add_argument("--early-stop-patience", type=int,   default=40)
    cfg, _ = ap.parse_known_args()

    global BASE_DIR
    if cfg.data_root:
        BASE_DIR = cfg.data_root
    else:
        for cand in [
            "/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge",
            "/kaggle/input/shift-guard-10-robust-image-classification-challenge",
            "shift-guard-10-robust-image-classification-challenge",
            os.path.join(os.getcwd(), "shift-guard-10-robust-image-classification-challenge"),
        ]:
            if os.path.isdir(cand):
                BASE_DIR = cand
                break

    if not os.path.isdir(BASE_DIR):
        print(f"ERROR: data not found at {BASE_DIR}")
        sys.exit(1)

    if cfg.debug:
        cfg.epochs = 2; cfg.batch_size = 32; cfg.swa_start = 9999
        cfg.tta = 2;    cfg.seeds = [42];    cfg.early_stop_patience = 9999

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        print(f"  GPUs: {torch.cuda.device_count()}")

    ref  = CifarDataset(BASE_DIR, "train", holdout=cfg.val_ratio, seed=42)
    freq = ref.class_counts()
    del ref

    wts_list = []
    for s in cfg.seeds:
        fp = f"checkpoints/final_{s}.pth"
        if ckpt_ok(fp):
            print(f"  Loading {fp}")
            wts_list.append(torch.load(fp, map_location=device, weights_only=True))
        else:
            if os.path.isfile(fp):
                print(f"  Corrupt final ckpt — retraining seed {s}")
                os.remove(fp)
            w, _ = train_model(s, cfg, device, freq)
            wts_list.append(w)

    test_ds = CifarDataset(BASE_DIR, "test", eval_tf())
    ens     = None
    for wts in wts_list:
        m = WideNet(depth=28, widen=10, nc=NUM_CLASSES, drop=0.3).to(device)
        if torch.cuda.device_count() > 1:
            m = torch.nn.DataParallel(m)
        raw = m.module if isinstance(m, torch.nn.DataParallel) else m
        raw.load_state_dict(wts)
        m.eval()

        p, ids = run_tta(m, test_ds, device, views=cfg.tta, bs=512)
        ens = p if ens is None else ens + p
        del m; torch.cuda.empty_cache()

    ens /= len(wts_list)
    make_submission(ens, ids, os.path.join(OUT_DIR, cfg.output))


if __name__ == "__main__":
    main()


  GPUs: 2

────────────────────────────────────────────────────────────
  Seed 42 | Epochs 109
────────────────────────────────────────────────────────────

  No checkpoint found — starting fresh.
  Ep  1/109 | Tr 1.934/7.8% | Va 1.454/18.0% F1=0.1734 | lr=0.000600 | 67.7s
  Ep  2/109 | Tr 1.802/11.3% | Va 1.513/29.3% F1=0.2373 | lr=0.000900 | 60.9s
  Ep  3/109 | Tr 1.716/15.2% | Va 1.467/26.3% F1=0.2478 | lr=0.001200 | 60.4s
  Ep  4/109 | Tr 1.652/20.3% | Va 1.693/23.5% F1=0.2240 | lr=0.001500 | 60.7s
  Ep  5/109 | Tr 1.606/23.0% | Va 1.227/29.0% F1=0.2812 | lr=0.001500 | 61.4s
  Ep  6/109 | Tr 1.548/27.9% | Va 1.427/39.3% F1=0.3562 | lr=0.001500 | 61.4s
  Ep  7/109 | Tr 1.505/29.7% | Va 1.296/41.2% F1=0.3504 | lr=0.001499 | 62.5s
  Ep  8/109 | Tr 1.481/31.6% | Va 1.706/36.5% F1=0.3211 | lr=0.001497 | 62.0s
  Ep  9/109 | Tr 1.459/33.7% | Va 1.191/45.1% F1=0.4072 | lr=0.001495 | 63.6s
  Ep 10/109 | Tr 1.412/36.9% | Va 2.037/36.9% F1=0.4050 | lr=0.001491 | 62.5s
  Ep 11/109 | Tr 1.434/3